# Agricultural Disease Detection System - Training Pipeline
## Deep Learning Model Training with Explainable AI

This notebook implements a complete training pipeline for plant disease detection using the PlantDoc dataset.

**Objectives:**
1. Load and preprocess PlantDoc dataset
2. Build CNN architecture for disease classification
3. Train model with data augmentation
4. Evaluate model performance
5. Apply SHAP for explainability
6. Save models for deployment

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0, ResNet50, VGG16

# ML Libraries
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Dataset Configuration & Exploration

In [ ]:
# Dataset paths
BASE_PATH = Path('c:/Users/Siri Nenta/OneDrive/Desktop/agri/PlantDoc-Dataset')
TRAIN_PATH = BASE_PATH / 'train'
TEST_PATH = BASE_PATH / 'test'

# Create output directories for models
OUTPUT_DIR = Path('models')
OUTPUT_DIR.mkdir(exist_ok=True)

# Configuration
IMG_SIZE = 224  # Standard size for transfer learning models
BATCH_SIZE = 32
EPOCHS = 50
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")
print(f"Training directory exists: {TRAIN_PATH.exists()}")
print(f"Test directory exists: {TEST_PATH.exists()}")

In [ ]:
# Get class information
train_classes = sorted([d.name for d in TRAIN_PATH.iterdir() if d.is_dir()])
test_classes = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])

num_classes = len(train_classes)

print(f"Number of classes: {num_classes}")
print(f"\nDisease Classes:")
for i, cls in enumerate(train_classes, 1):
    print(f"{i:2d}. {cls}")

In [ ]:
# Dataset distribution analysis
train_counts = {}
test_counts = {}

for cls in train_classes:
    train_dir = TRAIN_PATH / cls
    test_dir = TEST_PATH / cls
    train_counts[cls] = len(list(train_dir.glob('*.*')))
    test_counts[cls] = len(list(test_dir.glob('*.*')))

# Create distribution dataframe
dist_df = pd.DataFrame({
    'Class': list(train_counts.keys()),
    'Train': list(train_counts.values()),
    'Test': list(test_counts.values())
})

dist_df['Total'] = dist_df['Train'] + dist_df['Test']

print("\nDataset Distribution:")
print(dist_df.to_string())
print(f"\nTotal Training Images: {dist_df['Train'].sum()}")
print(f"Total Test Images: {dist_df['Test'].sum()}")
print(f"Total Images: {dist_df['Total'].sum()}")

## 3. Data Loading & Preprocessing

In [ ]:
def load_images_from_directory(directory_path, img_size=224, sample_size=None):
    """
    Load images and labels from directory structure
    Returns: images array, labels array, class names
    """
    images = []
    labels = []
    class_names = []
    
    classes = sorted([d.name for d in directory_path.iterdir() if d.is_dir()])
    class_to_idx = {cls: i for i, cls in enumerate(classes)}
    
    for cls_name in classes:
        cls_path = directory_path / cls_name
        img_files = list(cls_path.glob('*.*'))
        
        if sample_size:
            img_files = img_files[:sample_size]
        
        for img_file in img_files:
            try:
                img = Image.open(img_file).convert('RGB')
                img = img.resize((img_size, img_size), Image.Resampling.LANCZOS)
                images.append(np.array(img))
                labels.append(class_to_idx[cls_name])
            except Exception as e:
                print(f"Error loading {img_file}: {e}")
    
    return np.array(images), np.array(labels), classes

print("Loading training data...")
train_images, train_labels, class_names = load_images_from_directory(TRAIN_PATH, IMG_SIZE)
print(f"Train images shape: {train_images.shape}")
print(f"Train labels shape: {train_labels.shape}")

print("\nLoading test data...")
test_images, test_labels, _ = load_images_from_directory(TEST_PATH, IMG_SIZE)
print(f"Test images shape: {test_images.shape}")
print(f"Test labels shape: {test_labels.shape}")

In [ ]:
# Normalize images
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

# Convert labels to one-hot encoding
from tensorflow.keras.utils import to_categorical
train_labels_encoded = to_categorical(train_labels, num_classes)
test_labels_encoded = to_categorical(test_labels, num_classes)

print(f"Training data normalized")
print(f"Train images range: [{train_images.min():.2f}, {train_images.max():.2f}]")
print(f"Train labels encoded shape: {train_labels_encoded.shape}")
print(f"Test labels encoded shape: {test_labels_encoded.shape}")

## 4. Data Augmentation

In [ ]:
# Create data augmentation pipeline
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2]
)

test_datagen = ImageDataGenerator()

print("Data augmentation pipelines created")
print("Training augmentation includes:")
print("  - Rotation: ±20°")
print("  - Horizontal flip")
print("  - Width/Height shift: ±20%")
print("  - Zoom: ±20%")
print("  - Brightness: 80%-120%")

## 5. Build CNN Architecture

In [ ]:
def build_transfer_learning_model(input_shape, num_classes, base_model_name='efficientnet'):
    """
    Build transfer learning model for disease classification
    """
    if base_model_name.lower() == 'efficientnet':
        base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    elif base_model_name.lower() == 'resnet':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Freeze base model weights
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

print("Building transfer learning model with EfficientNetB0...")
model, base_model = build_transfer_learning_model(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=num_classes,
    base_model_name='efficientnet'
)

model.summary()

## 6. Model Compilation & Training

In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print("Model compiled successfully")

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    str(OUTPUT_DIR / 'best_disease_detection_model.h5'),
    monitor='val_accuracy',
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

print("Callbacks configured")

In [ ]:
# Train model
print("\nStarting model training...")
print(f"Total epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")

history = model.fit(
    train_datagen.flow(train_images, train_labels_encoded, batch_size=BATCH_SIZE),
    validation_data=(test_images, test_labels_encoded),
    epochs=EPOCHS,
    callbacks=[early_stop, model_checkpoint, reduce_lr],
    verbose=1
)

print("\nTraining completed!")

## 7. Model Evaluation

In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(
    test_images, test_labels_encoded, verbose=0
)

print(f"\nTest Results:")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
f1 = (2 * (test_precision * test_recall) / (test_precision + test_recall)) if (test_precision + test_recall) > 0 else 0.0
print(f"F1-Score: {f1:.4f}")

In [ ]:
# Make predictions
print("Making predictions on test set...")
y_pred = model.predict(test_images)
y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(test_labels_encoded, axis=1)

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_true_labels, y_pred_labels,
    target_names=class_names,
    digits=3
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true_labels, y_pred_labels)

# Plot confusion matrix
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Disease Detection Model', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved")

## 8. Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train')
axes[0, 0].plot(history.history['val_accuracy'], label='Test')
axes[0, 0].set_title('Model Accuracy', fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train')
axes[0, 1].plot(history.history['val_loss'], label='Test')
axes[0, 1].set_title('Model Loss', fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history.get('precision', history.history.get('precision_1', [])), label='Train')
axes[1, 0].plot(history.history.get('val_precision', history.history.get('val_precision_1', [])), label='Test')
axes[1, 0].set_title('Model Precision', fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history.get('recall', history.history.get('recall_1', [])), label='Train')
axes[1, 1].plot(history.history.get('val_recall', history.history.get('val_recall_1', [])), label='Test')
axes[1, 1].set_title('Model Recall', fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_history.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Training history plot saved")

## 9. Feature Importance & Explainability (SHAP)

In [ ]:
# Sample test images for SHAP analysis
sample_indices = np.random.choice(len(test_images), 10, replace=False)
sample_images = test_images[sample_indices]
sample_labels = test_labels[sample_indices]

# Make predictions on samples
sample_predictions = model.predict(sample_images)
sample_pred_labels = np.argmax(sample_predictions, axis=1)

# Display predictions
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()

for idx, ax in enumerate(axes):
    ax.imshow(sample_images[idx])
    true_class = class_names[sample_labels[idx]]
    pred_class = class_names[sample_pred_labels[idx]]
    confidence = sample_predictions[idx][sample_pred_labels[idx]]
    
    color = 'green' if sample_labels[idx] == sample_pred_labels[idx] else 'red'
    ax.set_title(f'True: {true_class}\nPred: {pred_class} ({confidence:.2f})', 
                 color=color, fontweight='bold', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'sample_predictions.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Sample predictions visualization saved")

## 10. Model Persistence & Export

In [ ]:
# Save complete model
model_path = OUTPUT_DIR / 'disease_detection_model.keras'
model.save(str(model_path))
print(f"Model saved: {model_path}")

# Save model in H5 format
h5_path = OUTPUT_DIR / 'disease_detection_model.h5'
model.save(str(h5_path))
print(f"Model saved (H5): {h5_path}")

# Save class names
import json
class_mapping = {i: name for i, name in enumerate(class_names)}
with open(str(OUTPUT_DIR / 'class_names.json'), 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"Class names saved")

# Save model metrics
metrics = {
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'num_classes': num_classes,
    'image_size': IMG_SIZE,
    'total_parameters': model.count_params()
}

with open(str(OUTPUT_DIR / 'model_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Model metrics saved")

print(f"\nAll models exported to: {OUTPUT_DIR}")

## 11. Create Inference Pipeline

In [ ]:
class DiseaseDetectionModel:
    """Wrapper class for disease detection inference"""
    
    def __init__(self, model_path, class_names_path, img_size=224):
        self.model = keras.models.load_model(model_path)
        self.img_size = img_size
        
        with open(class_names_path, 'r') as f:
            self.class_mapping = json.load(f)
            self.class_names = [self.class_mapping[str(i)] for i in range(len(self.class_mapping))]
    
    def predict_from_path(self, image_path, top_k=3):
        """Predict disease from image path"""
        img = Image.open(image_path).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.Resampling.LANCZOS)
        img_array = np.array(img).astype('float32') / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        predictions = self.model.predict(img_array, verbose=0)[0]
        
        # Get top-k predictions
        top_indices = np.argsort(predictions)[-top_k:][::-1]
        results = []
        for idx in top_indices:
            results.append({
                'disease': self.class_names[idx],
                'confidence': float(predictions[idx])
            })
        
        return results
    
    def predict_from_array(self, image_array, top_k=3):
        """Predict disease from numpy array"""
        if image_array.shape[-1] != 3:
            image_array = cv2.cvtColor(image_array, cv2.COLOR_GRAY2BGR)
        
        img = Image.fromarray((image_array * 255).astype('uint8')) if image_array.max() <= 1 else Image.fromarray(image_array.astype('uint8'))
        img = img.resize((self.img_size, self.img_size), Image.Resampling.LANCZOS)
        img_array = np.array(img).astype('float32') / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        predictions = self.model.predict(img_array, verbose=0)[0]
        
        top_indices = np.argsort(predictions)[-top_k:][::-1]
        results = []
        for idx in top_indices:
            results.append({
                'disease': self.class_names[idx],
                'confidence': float(predictions[idx])
            })
        
        return results

# Test inference pipeline
print("Instantiating disease detection model...")
detector = DiseaseDetectionModel(
    str(OUTPUT_DIR / 'disease_detection_model.keras'),
    str(OUTPUT_DIR / 'class_names.json'),
    img_size=IMG_SIZE
)
print("Disease detection model ready for inference!")

In [ ]:
# Test with sample images
print("Testing inference on random test samples...\n")
for i in range(5):
    img_idx = np.random.choice(len(sample_images))
    predictions = detector.predict_from_array(sample_images[img_idx], top_k=3)
    true_class = class_names[sample_labels[img_idx]]
    
    print(f"Sample {i+1}:")
    print(f"  True Disease: {true_class}")
    print(f"  Predictions:")
    for j, pred in enumerate(predictions, 1):
        print(f"    {j}. {pred['disease']}: {pred['confidence']:.4f}")
    print()

## 12. Model Summary & Next Steps

In [ ]:
print("="*70)
print("DISEASE DETECTION MODEL - TRAINING SUMMARY")
print("="*70)
print(f"\nDataset:")
print(f"  - Total Images: {len(train_images) + len(test_images)}")
print(f"  - Training Images: {len(train_images)}")
print(f"  - Test Images: {len(test_images)}")
print(f"  - Number of Disease Classes: {num_classes}")

print(f"\nModel Architecture:")
print(f"  - Base Model: EfficientNetB0")
print(f"  - Transfer Learning: Yes (frozen base)")
print(f"  - Total Parameters: {model.count_params():,}")
print(f"  - Input Size: {IMG_SIZE}x{IMG_SIZE}")

print(f"\nTraining Configuration:")
print(f"  - Epochs: {len(history.history['loss'])}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Data Augmentation: Yes")
print(f"  - Early Stopping: Yes (patience=10)")

print(f"\nPerformance Metrics:")
print(f"  - Accuracy: {test_accuracy:.4f}")
print(f"  - Loss: {test_loss:.4f}")
print(f"  - Precision: {test_precision:.4f}")
print(f"  - Recall: {test_recall:.4f}")
f1 = (2 * (test_precision * test_recall) / (test_precision + test_recall)) if (test_precision + test_recall) > 0 else 0.0
print(f"  - F1-Score: {f1:.4f}")

print(f"\nExported Models:")
print(f"  - {OUTPUT_DIR / 'disease_detection_model.keras'}")
print(f"  - {OUTPUT_DIR / 'disease_detection_model.h5'}")
print(f"  - {OUTPUT_DIR / 'class_names.json'}")
print(f"  - {OUTPUT_DIR / 'model_metrics.json'}")
print(f"  - {OUTPUT_DIR / 'confusion_matrix.png'}")
print(f"  - {OUTPUT_DIR / 'training_history.png'}")
print(f"  - {OUTPUT_DIR / 'sample_predictions.png'}")

print(f"\nNext Steps:")
print(f"  1. Build Flask REST API for model inference")
print(f"  2. Implement SHAP explainability for predictions")
print(f"  3. Create web interface for disease detection")
print(f"  4. Integrate environmental data APIs")
print(f"  5. Build crop yield and pest detection models")
print(f"  6. Develop expert advisory framework")
print(f"  7. Create multi-language chatbot interface")
print("="*70)